# Week 4 - Linear regression baseline

Fit an ordinary least-squares `LinearRegression` model to predict California single-family `ClosePrice`, then measure R² on the held-out test set. This is the Week 4 linear baseline; the repository also contains a separate gradient-boosted valuation evaluation.

Input: `csv/cleaned_sales.csv`, produced by `02_preprocessing.ipynb`. Week 3 selected the preceding 12 training months using validation months before the test period and exported 41 finite numeric model columns. Its imputers and category encoders were fitted on training data; the test rows were transformed without using their labels. The CSV is local and ignored by Git. Run this notebook from the repository root after generating that file.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

DATA_PATH = Path('csv/cleaned_sales.csv')
if not DATA_PATH.is_file():
    raise FileNotFoundError('Run 02_preprocessing.ipynb from the repository root to create csv/cleaned_sales.csv.')

## Use the Week 3 split

`DataSplit` fixes the training and test rows. `ClosePrice` is the label; `CloseDate` and `DataSplit` are audit metadata. All three are excluded from the model input. Check that every remaining column is numeric and finite and that training dates precede test dates.

In [2]:
data = pd.read_csv(DATA_PATH, parse_dates=['CloseDate'])
metadata_columns = ['CloseDate', 'ClosePrice', 'DataSplit']
missing_columns = set(metadata_columns) - set(data.columns)
if missing_columns:
    raise ValueError(f'Missing required columns: {sorted(missing_columns)}')
if set(data['DataSplit']) != {'train', 'test'}:
    raise ValueError('DataSplit must contain exactly train and test.')

feature_columns = [column for column in data.columns if column not in metadata_columns]
if not feature_columns:
    raise ValueError('No model features found.')
non_numeric = [column for column in feature_columns
               if not pd.api.types.is_numeric_dtype(data[column])]
if non_numeric:
    raise ValueError(f'Model features must be numeric: {non_numeric}')

X = data.loc[:, feature_columns]
y = pd.to_numeric(data['ClosePrice'], errors='raise')
if not np.isfinite(X.to_numpy(dtype=float)).all():
    raise ValueError('Model features must all be finite.')
if not np.isfinite(y.to_numpy(dtype=float)).all() or not y.gt(0).all():
    raise ValueError('ClosePrice labels must be finite and positive.')
if data['CloseDate'].isna().any():
    raise ValueError('Every row needs a valid CloseDate.')

train_mask = data['DataSplit'].eq('train')
test_mask = data['DataSplit'].eq('test')
if data.loc[train_mask, 'CloseDate'].max() >= data.loc[test_mask, 'CloseDate'].min():
    raise ValueError('Training dates must precede test dates.')

X_train, X_test = X.loc[train_mask], X.loc[test_mask]
y_train, y_test = y.loc[train_mask], y.loc[test_mask]
split_audit = data.groupby('DataSplit', sort=False).agg(
    rows=('ClosePrice', 'size'),
    first_close=('CloseDate', 'min'),
    last_close=('CloseDate', 'max'),
)
print(f'Model features: {len(feature_columns)}')
display(split_audit)

Model features: 41


,rows,first_close,last_close
DataSplit,,,
train,130274,2025-04-01,2026-03-31
test,12470,2026-04-01,2026-04-30


## Fit and score the baseline

Fit only on `train` rows. Score the unmodified predictions against `test` labels once, using R² as the requested headline metric. MAE and RMSE give the dollar scale of the errors. Ordinary linear regression does not constrain predictions to positive prices.

In [3]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
test_predictions = baseline.predict(X_test)

test_r2 = r2_score(y_test, test_predictions)
test_mae = mean_absolute_error(y_test, test_predictions)
test_rmse = np.sqrt(mean_squared_error(y_test, test_predictions))
negative_predictions = int(np.sum(test_predictions < 0))

baseline_results = pd.DataFrame({
    'Metric': ['Test R²', 'Test MAE', 'Test RMSE', 'Negative test predictions'],
    'Result': [f'{test_r2:.4f}', f'${test_mae:,.0f}',
               f'${test_rmse:,.0f}', f'{negative_predictions:,} / {len(X_test):,}'],
})
display(baseline_results)

,Metric,Result
0,Test R²,0.6632
1,Test MAE,"$389,746"
2,Test RMSE,"$786,908"
3,Negative test predictions,"623 / 12,470"


## Recorded baseline result

For the Week 3 export in this workspace (130,274 training sales from April 2025 through March 2026; 12,470 test sales from April 2026), the **test R² is 0.6632**. Test MAE is **$389,746** and RMSE is **$786,908**. The unconstrained linear model predicts a negative price for **623 of 12,470** test homes. These are raw baseline scores, with no clipping or postprocessing.

R² measures fit to held-out closed MLS sales. It does not establish accuracy for off-market homes; that coverage requires a separate evaluation.